# Making the model with time in the input


In [ ]:
# number of epochs with no improvement after which training will be stopped
EARLY_STOPPING = 100
RETUNE_BY_OVERWRITING = False


# the folder with the training/validation/test datasets
DATASET_FOLDER = "../../data/datasets"
TUNER_RESULTS_FOLDER = "../../data/tuner_results"


## Setting up parallelism

In [ ]:
import os

NUMBER_OF_PHYSICAL_CODES = 10

def setup_threading(n_physical_cores):
    if not isinstance(n_physical_cores, int) or n_physical_cores < 1:
        raise ValueError("Number of physical cores must be a positive integer")
    omp_n_cores = n_physical_cores
    intra_cores = n_physical_cores
    inter_cores = max(1, n_physical_cores // 4)

    os.environ["OMP_NUM_THREADS"] = str(omp_n_cores)
    os.environ["TF_NUM_INTRAOP_THREADS"] = str(intra_cores)
    os.environ["TF_NUM_INTEROP_THREADS"] = str(inter_cores)

    os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # Hide INFO and WARNING messages


## Load the necessary modules and data

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.metrics import accuracy_score, r2_score
import tensorflow as tf
from keras_tuner import Hyperband
from neuralns import mnn


data = mnn.read_datasets(DATASET_FOLDER)

In [ ]:
TUNING_PARAMS = {
    "n_hidden": [2, 4, 8],
    "n_neurons": [16, 32, 64, 128],
    "learning_rate": [0.01, 0.001, 0.0001],
    "dropout": [0.0],           # it was found ineffective
    "l1": [0.0, 1.0e-6, 1.0e-7, 1.0e-8],
    "l2": [0.0],                # it was found ineffective
    "activation": ["relu"],     # `sigmoid`` was found ineffective
    "reduce_lr_patience": [0],  # it was found ineffective
}

tr = mnn.InputOutputTransformer(
    x_features=['logL', 'logP', 'logPdot', 'logt'],
    y_reg_vars= ['logB', 'M', 'logB0'],
    y_cls_var='env=H', data=data['train'])

def build_model(hp):
    def choice(parameter_name):
        return hp.Choice(parameter_name, TUNING_PARAMS[parameter_name])

    return mnn.MultioutNN(
        transformer=tr,
        n_hidden=choice("n_hidden"),
        n_neurons=choice("n_neurons"),
        learning_rate=choice("learning_rate"),
        dropout=choice("dropout"),
        reduce_lr_patience=choice("reduce_lr_patience"),
        l1=choice("l1"),
        l2=choice("l2"),
        activation=choice("activation"),
        early_stopping=EARLY_STOPPING,
        verbose=False).model

tuner = Hyperband(build_model, objective="val_loss",
                  max_epochs=30,
                  directory=TUNER_RESULTS_FOLDER,
                  project_name="with",
                  hyperband_iterations=1,
                  overwrite=RETUNE_BY_OVERWRITING,
                 )

## Start the hyperparameter optimization

In [ ]:
tuner.search(
    x=tr.transform("X", data["train"]),
    y=[tr.transform("Y_reg", data["train"]), tr.transform("Y_cls", data["train"])],
    batch_size=64,
    validation_data=[tr.transform("X", data["valid"]), [tr.transform("Y_reg", data["valid"]), tr.transform("Y_cls", data["valid"])]],
    verbose=True,
    epochs=100,
)

## Visualize the performance of the optimal model

In [ ]:
best_model = tuner.get_best_models(num_models=1)[0]
best_mnn = mnn.MultioutNN(transformer=tr, verbose=False)
best_mnn.model = best_model
# best_model.
Y_reg_pred, Y_cls_pred = best_mnn.predict(data["test"])

y_cls_var = best_mnn.transformer.y_cls_var
acc = accuracy_score(data["test"][y_cls_var], Y_cls_pred)
print(f"Classification accuracy of `{y_cls_var}`: {acc:.4g}")

for i, var in enumerate(best_mnn.transformer.y_reg_vars):
    y_reg_pred = Y_reg_pred[var]
    y_reg_true = data["test"][var]
    r2 = r2_score(y_reg_true, y_reg_pred)
    print(f"R^2 score of prediction of `{var}`: {r2:.4g}")

    y_min = min(min(y_reg_true), min(y_reg_pred))
    y_max = max(max(y_reg_true), max(y_reg_pred))

    yy = np.linspace(y_min, y_max, 31)

    plt.figure()
    plt.plot(y_reg_true, y_reg_pred, "k.", alpha=0.5)
    plt.plot(yy, yy, "r--", alpha=0.7)
    plt.title(var)
    plt.xlabel("True")
    plt.ylabel("Predicted")
plt.show()

## Export tuner results in CSV format

In [ ]:

results = []

for trial_id, trial in tuner.oracle.trials.items():
    entry = trial.hyperparameters.values.copy()   # hyperparameters dict
    entry['score'] = trial.score                  # objective score
    results.append(entry)

results = pd.DataFrame(results)
results = results.sort_values("score", ascending=True)
results.to_csv(os.path.join(tuner.project_dir, "results.csv"), index_label="trial_id")
results

## Get weights of best model, and report optimal hyperparameters

In [ ]:
best_hyper = tuner.get_best_hyperparameters()[0]
best_hyper = best_hyper.values
best_hyper = {key: value for key, value in best_hyper.items() if not key.startswith("tuner/")}
best_model = tuner.get_best_models()[0]
best_weights = best_model.weights
del best_model
best_hyper

## Refit the best model to obtain the final model

In [ ]:
best_mnn = mnn.MultioutNN(transformer=tr, early_stopping=EARLY_STOPPING, **best_hyper)
best_mnn.model.set_weights(best_weights)
best_mnn.fit(training_data=data["train"], validation_data=data["valid"], epochs=10000)
best_mnn.save(os.path.join(tuner.project_dir, "refit_best_model.mod"))

 90/228 ━━━━━━━━━━━━━━━━━━━━ 0s 3ms/step - clsout_accuracy: 0.8435 - clsout_loss: 0.3054 - loss: 0.4684 - regout_loss: 0.0530 - regout_mean_absolute_error: 0.0530